### PEPit code for OFW (fixed parameters)

In [ ]:
import numpy as np
from PEPit import PEP
from PEPit.functions import ConvexLipschitzFunction
from PEPit.functions import ConvexIndicatorFunction
from PEPit.primitive_steps import linear_optimization_step

# from PEPit.primitive_steps import proximal_step
# from PEPit.examples.online_learning.online_frank_wolfe import wc_online_frank_wolfe

In [ ]:
def PEPit_OFW_regret(n, M=1, D=1, eta=None, sigma=None,
                     parameter_choice=0, return_theoretical_tau=True,
                     verbose=0, wrapper="cvxpy", solver=None):
    '''Using PEPit, computes the worst-case regret of the online Frank-Wolfe algorithm for n steps, 
        for M-Lipschitz convex functions and a convex set with diameter D.

        - eta and sigma are the step-sizes of the algorithm. If None, they are set according to the parameter_choice argument (see below).
            - parameter_choice = 0 corresponds to the step-sizes choice of Theorem 3.1 in (Weibel et al., 2026).
            - parameter_choice = 1 corresponds to the anytime step-sizes choice of Theorem 3.3 in (Weibel et al., 2026).
            - parameter_choice = 2 corresponds to the fixed step-sizes choice of Theorem 7.3 in (Hazan, 2016).
        - When return_theoretical_tau is True, returns a tuple (pepit_tau, theoretical_tau) where pepit_tau is 
            the worst-case guarantee returned by PEPit and theoretical_tau is the theoretical guarantee corresponding 
            to the chosen step-sizes and Theorem. Otherwise, returns only pepit_tau.
        - verbose, wrapper and solver are arguments passed to the PEPit solver. See PEPit documentation for more details.
    '''

    if eta is None:
        if parameter_choice == 1:
            eta = [D / (2 * M) * (3 / k) ** (3 / 4) for k in range(1, n + 1)]
        elif parameter_choice == 2:
            eta = [D / (2 * M) * n ** (3 / 4)] * n
        else:
            eta = [D / (2 * M) * (3 / n) ** (3 / 4)] * n
    elif isinstance(eta, (int, float)):
        eta = [eta] * n
    if sigma is None:
        if parameter_choice == 1:
            sigma = [min([1, np.sqrt(3 / k)]) for k in range(1, n + 1)]
        elif parameter_choice == 2:
            sigma = [min([1, 2 * np.sqrt(1 / n)])] * n
        else:
            sigma = [min([1, np.sqrt(3 / n)])] * n
    elif isinstance(sigma, (int, float)):
        sigma = [sigma] * n

    # Instantiate PEP
    problem = PEP()

    # Declare a sequence of M-Lipschitz convex functions fi and an indicator function with Diameter D
    fis = [problem.declare_function(ConvexLipschitzFunction, M=M) for _ in range(n)]
    h = problem.declare_function(function_class=ConvexIndicatorFunction, D=D)

    # Defining a reference point
    xRef = problem.set_initial_point()
    _, _ = h.oracle(xRef)  # insures that the reference point is in the domain of h

    # Then define the starting point x0 of the algorithm
    x1 = problem.set_initial_point()
    _, _ = h.oracle(x1)  # insures that the initial point is in the domain of h

    # Run n steps of the online Conditional Gradient / Frank-Wolfe method starting from x1
    x = x1
    acc_g = 0 * xRef
    regret = 0

    for i in range(n):
        g, f = fis[i].oracle(x)
        regret = regret + f - fis[i](xRef)
        acc_g = acc_g + g
        dir_t = (x - x1) + eta[i] * acc_g
        v, _, _ = linear_optimization_step(dir_t, h)
        x = (1 - sigma[i]) * x + sigma[i] * v

    # Set the performance metric to the function values accuracy
    problem.set_performance_metric(regret)

    # Solve the PEP
    pepit_verbose = max(verbose, 0)
    pepit_tau = problem.solve(wrapper=wrapper, solver=solver, verbose=pepit_verbose)

    # Compute theoretical guarantee
    if parameter_choice == 1:
        theoretical_tau = 5 / 3 ** (3 / 4) * M * D * n ** (3 / 4)
    elif parameter_choice == 2:
        theoretical_tau = 8 * M * D * n ** (3 / 4)
    else:
        theoretical_tau = 4 / 3 ** (3 / 4) * M * D * n ** (3 / 4)

    # Print conclusion if required
    if verbose != 0:
        print('*** Worst-case regret of online Frank-Wolfe ***')
        print('\tPEPit guarantee:\t R_n <= {:.6}'.format(pepit_tau))
        print('\tTheoretical guarantee:\t R_n <= {:.6}'.format(theoretical_tau))

    # Return the worst-case guarantee of the evaluated method (and the reference theoretical value)
    if return_theoretical_tau:
        return pepit_tau, theoretical_tau
    else:
        return pepit_tau

In [18]:
pepit_tau, theoretical_tau = PEPit_OFW_regret(n=10, M=1, D=1, verbose=0, parameter_choice=0)
print('PEPit guarantee:', pepit_tau)
print('Theoretical guarantee:', theoretical_tau)

PEPit guarantee: 6.661196156384808
Theoretical guarantee: 9.867770726563805


In [61]:
def PEPit_general_OFW_regret(n, M=1, D=1, eta=None, beta=None, sigma=None, gamma=None,
                                parameter_choice=0, return_theoretical_tau=True,
                                verbose=0, wrapper="cvxpy", solver=None):
    '''Using PEPit, computes the worst-case regret of the general online Frank-Wolfe algorithm
        (from Equation (1) in (Weibel et al., 2026)) for n steps, 
        for M-Lipschitz convex functions and a convex set with diameter D.

        - eta, beta, and gamma (and sigma) are the parameters of the algorithm. 
            - If gamma is None, it is set as a recursive udpate using sigma (similar to simple online Frank-Wolfe).
            - If gamma is not None, sigma is ignored.
            - If beta is None, it is set equal to gamma, corresponding to linearization at the current iterate.
            - If eta or sigma is None, they are set according to the parameter_choice argument (see below).
                - parameter_choice = 0 corresponds to the step-sizes choice of Theorem 3.1 in (Weibel et al., 2026).
                - parameter_choice = 1 corresponds to the anytime step-sizes choice of Theorem 3.3 in (Weibel et al., 2026).
                - parameter_choice = 2 corresponds to the fixed step-sizes choice of Theorem 7.3 in (Hazan, 2016).
            - Otherwise, they should be Numpy arrays of size n x n.
        - When return_theoretical_tau is True, returns a tuple (pepit_tau, theoretical_tau) where pepit_tau is 
            the worst-case guarantee returned by PEPit and theoretical_tau is the theoretical guarantee corresponding 
            to the chosen step-sizes and Theorem. Otherwise, returns only pepit_tau.
        - verbose, wrapper and solver are arguments passed to the PEPit solver. See PEPit documentation for more details.
    '''

    ### Sets the parameters of the algorithm according to the parameter_choice argument and the user inputs

    if eta is None:
        if parameter_choice == 1:
            eta = np.zeros((n,n))
            for k in range(1, n + 1):
                eta[k-1,:] = D / (2 * M) * (3 / k) ** (3 / 4)
        elif parameter_choice == 2:
            eta = D / (2 * M) * n ** (3 / 4) * np.ones((n,n))
        else:
            eta = D / (2 * M) * (3 / n) ** (3 / 4) * np.ones((n,n))
    elif isinstance(eta, (int, float)):
        eta = [eta] * n
    elif isinstance(eta, (list, np.ndarray)):
        if len(eta) != n:
            raise ValueError('When eta is given as a list or a Numpy array, it should be of length n.')
        new_eta = np.zeros((n,n))
        for i in range(n):
            new_eta[i,:] = eta[i]
        eta = new_eta

    if (sigma is None) and (gamma is None):
        if parameter_choice == 1:
            sigma = [min([1, np.sqrt(3 / k)]) for k in range(1, n + 1)]
        elif parameter_choice == 2:
            sigma = [min([1, 2 * np.sqrt(1 / n)])] * n
        else:
            sigma = [min([1, np.sqrt(3 / n)])] * n

        gamma = np.zeros((n,n))
        for i in range(n):
            gamma[i,i-1] = sigma[i]
            for j in range(i-1):
                gamma[i,j] = (1-sigma[i]) * gamma[i-1,j]

    if beta is None:
        beta = gamma

    
    ### Now, creates the PEP and solve it

    # Instantiate PEP
    problem = PEP()

    # Declare a sequence of M-Lipschitz convex functions fi and an indicator function with Diameter D
    fis = [problem.declare_function(ConvexLipschitzFunction, M=M) for _ in range(n)]
    h = problem.declare_function(function_class=ConvexIndicatorFunction, D=D)

    # Defining a reference point
    xRef = problem.set_initial_point()
    _, _ = h.oracle(xRef)  # insures that the reference point is in the domain of h

    # Then define the starting point x0 of the algorithm
    x1 = problem.set_initial_point()
    _, _ = h.oracle(x1)  # insures that the initial point is in the domain of h

    # Run n steps of the general online Frank-Wolfe method starting from x1
    x = x1
    list_g = []
    list_v = []
    regret = 0

    for i in range(n):
        g, f = fis[i].oracle(x)
        list_g.append(g)
        regret = regret + f - fis[i](xRef)
        dir_t = sum( (eta[i,j] * list_g[j] for j in range(i+1)), 0*x1)
        dir_t = sum(( beta[i,j] * (list_v[j] - x1) for j in range(i) ), dir_t)
        v, _, _ = linear_optimization_step(dir_t, h)
        list_v.append(v)
        if i < n-1:
            x = x1 + sum((gamma[i+1,j] * (list_v[j] - x1) for j in range(i+1)), 0*x1)

    # Set the performance metric to the function values accuracy
    problem.set_performance_metric(regret)

    # Solve the PEP
    pepit_verbose = max(verbose, 0)
    pepit_tau = problem.solve(wrapper=wrapper, solver=solver, verbose=pepit_verbose)

    # Compute theoretical guarantee
    if parameter_choice == 1:
        theoretical_tau = 5 / 3 ** (3 / 4) * M * D * n ** (3 / 4)
    elif parameter_choice == 2:
        theoretical_tau = 8 * M * D * n ** (3 / 4)
    else:
        theoretical_tau = 4 / 3 ** (3 / 4) * M * D * n ** (3 / 4)

    # Print conclusion if required
    if verbose != 0:
        print('*** Worst-case regret of online Frank-Wolfe ***')
        print('\tPEPit guarantee:\t R_n <= {:.6}'.format(pepit_tau))
        print('\tTheoretical guarantee:\t R_n <= {:.6}'.format(theoretical_tau))

    # Return the worst-case guarantee of the evaluated method (and the reference theoretical value)
    if return_theoretical_tau:
        return pepit_tau, theoretical_tau
    else:
        return pepit_tau

In [62]:
pepit_tau, theoretical_tau = PEPit_OFW_regret(n=10, M=1, D=1, verbose=0, parameter_choice=0)
print('PEPit guarantee:', pepit_tau)
print('Theoretical guarantee:', theoretical_tau)

pepit_tau, theoretical_tau = PEPit_general_OFW_regret(n=10, M=1, D=1, verbose=0, parameter_choice=0)
print('PEPit guarantee:', pepit_tau)
print('Theoretical guarantee:', theoretical_tau)

PEPit guarantee: 6.661196156384808
Theoretical guarantee: 9.867770726563805
PEPit guarantee: 6.661196156385865
Theoretical guarantee: 9.867770726563805


In [ ]:
import time

n_values = [n for n in range(1, 11)] + [n for n in range(10,101,5)]
results = []
times = []

for n in n_values:
    start_time = time.time()
    result = PEPit_OFW_regret(n=n, M=1, D=1, verbose=0, parameter_choice=0)
    elapsed_time = time.time() - start_time

    results.append(result)
    times.append(elapsed_time)

    print(f"n = {n}: result = {result}, time = {elapsed_time:.6f} seconds")

# Now you can use `results` and `times` for further analysis

n = 1: result = (0.9999999885341219, 1.7547653506033234), time = 0.051107 seconds
n = 2: result = (1.8660253802931281, 2.9511517858675242), time = 0.067859 seconds
n = 3: result = (2.7950850363262085, 4.0), time = 0.109990 seconds
n = 4: result = (3.4169499528885416, 4.963225915211199), time = 0.068076 seconds
n = 5: result = (4.005787596756963, 5.867411578622623), time = 0.080347 seconds
n = 6: result = (4.568277676530344, 6.727171322029717), time = 0.102238 seconds
n = 7: result = (5.109443465762224, 7.551662641322065), time = 0.134826 seconds
n = 8: result = (5.637524601789594, 8.347117760390866), time = 0.162611 seconds
n = 9: result = (6.155185726386532, 9.118028227819112), time = 0.203353 seconds
n = 10: result = (6.661196156384808, 9.867770726563805), time = 0.298690 seconds
n = 10: result = (6.661196156384808, 9.867770726563805), time = 0.271889 seconds
n = 15: result = (9.027558137681758, 13.374806099528442), time = 0.740620 seconds
n = 20: result = (11.190348047830701, 16.595

n = 1: result = (0.9999999885341219, 1.7547653506033234), time = 0.051107 seconds
n = 2: result = (1.8660253802931281, 2.9511517858675242), time = 0.067859 seconds
n = 3: result = (2.7950850363262085, 4.0), time = 0.109990 seconds
n = 4: result = (3.4169499528885416, 4.963225915211199), time = 0.068076 seconds
n = 5: result = (4.005787596756963, 5.867411578622623), time = 0.080347 seconds
n = 6: result = (4.568277676530344, 6.727171322029717), time = 0.102238 seconds
n = 7: result = (5.109443465762224, 7.551662641322065), time = 0.134826 seconds
n = 8: result = (5.637524601789594, 8.347117760390866), time = 0.162611 seconds
n = 9: result = (6.155185726386532, 9.118028227819112), time = 0.203353 seconds
n = 10: result = (6.661196156384808, 9.867770726563805), time = 0.298690 seconds
n = 10: result = (6.661196156384808, 9.867770726563805), time = 0.271889 seconds
n = 15: result = (9.027558137681758, 13.374806099528442), time = 0.740620 seconds
n = 20: result = (11.190348047830701, 16.59554606102609), time = 1.620886 seconds
n = 25: result = (13.217674378276067, 19.618873042551414), time = 3.351269 seconds
n = 30: result = (15.147741085882371, 22.493653007613965), time = 6.686959 seconds
n = 35: result = (17.000530424190043, 25.250505889183852), time = 14.170136 seconds
n = 40: result = (18.789599076102213, 27.91027038378948), time = 22.966939 seconds
n = 45: result = (20.5247907504991, 30.48796488927689), time = 40.762055 seconds
n = 50: result = (22.21386296726385, 32.99488002559844), time = 62.630188 seconds
n = 55: result = (23.86270497514305, 35.43978409331229), time = 94.350882 seconds
n = 60: result = (25.47592670183348, 37.82966436012703), time = 140.881171 seconds
n = 65: result = (27.057084289248433, 40.170206822580035), time = 221.846619 seconds
n = 70: result = (28.60917325258102, 42.46611977111502), time = 313.783617 seconds
n = 75: result = (30.13495597501673, 44.721359549995796), time = 533.602720 seconds
n = 80: result = (31.636691607770267, 46.939292628980986), time = 773.320634 seconds
n = 85: result = (33.11623951312971, 49.122814811261), time = 1202.683847 seconds
n = 90: result = (34.57528775595644, 51.27444076754809), time = 1603.325482 seconds
n = 95: result = (36.01503588418875, 53.39637251986722), time = 2126.254855 seconds

In [1]:
import re

data = """
n = 1: result = (0.9999999885341219, 1.7547653506033234), time = 0.051107 seconds
n = 2: result = (1.8660253802931281, 2.9511517858675242), time = 0.067859 seconds
n = 3: result = (2.7950850363262085, 4.0), time = 0.109990 seconds
n = 4: result = (3.4169499528885416, 4.963225915211199), time = 0.068076 seconds
n = 5: result = (4.005787596756963, 5.867411578622623), time = 0.080347 seconds
n = 6: result = (4.568277676530344, 6.727171322029717), time = 0.102238 seconds
n = 7: result = (5.109443465762224, 7.551662641322065), time = 0.134826 seconds
n = 8: result = (5.637524601789594, 8.347117760390866), time = 0.068076 seconds
n = 9: result = (6.155185726386532, 9.118028227819112), time = 0.203353 seconds
n = 10: result = (6.661196156384808, 9.867770726563805), time = 0.298690 seconds
n = 10: result = (6.661196156384808, 9.867770726563805), time = 0.271889 seconds
n = 15: result = (9.027558137681758, 13.374806099528442), time = 0.740620 seconds
n = 20: result = (11.190348047830701, 16.59554606102609), time = 1.620886 seconds
n = 25: result = (13.217674378276067, 19.618873042551414), time = 3.351269 seconds
n = 30: result = (15.147741085882371, 22.493653007613965), time = 6.686959 seconds
n = 35: result = (17.000530424190043, 25.250505889183852), time = 14.170136 seconds
n = 40: result = (18.789599076102213, 27.91027038378948), time = 22.966939 seconds
n = 45: result = (20.5247907504991, 30.48796488927689), time = 40.762055 seconds
n = 50: result = (22.21386296726385, 32.99488002559844), time = 62.630188 seconds
n = 55: result = (23.86270497514305, 35.43978409331229), time = 94.350882 seconds
n = 60: result = (25.47592670183348, 37.829664360127035), time = 140.881171 seconds
n = 65: result = (27.057084289248433, 40.170206822580035), time = 221.846619 seconds
n = 70: result = (28.60917325258102, 42.46611977111502), time = 313.783617 seconds
n = 75: result = (30.13495597501673, 44.721359549995796), time = 533.602720 seconds
n = 80: result = (31.636691607770267, 46.939292628980986), time = 773.320634 seconds
n = 85: result = (33.11623951312971, 49.122814811261), time = 1202.683847 seconds
n = 90: result = (34.57528775595644, 51.27444076754809), time = 1603.325482 seconds
n = 95: result = (36.01503588418875, 53.39637251986722), time = 2126.254855 seconds
"""

# Extract n and time using regex
pattern = r"n = (\d+):.*time = ([\d.]+) seconds"
matches = re.findall(pattern, data)

# Separate n and time into lists
n_values = [int(match[0]) for match in matches]
time_values = [float(match[1]) for match in matches]

# Print the extracted values
print("n values:", n_values)
print("time values:", time_values)

n values: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95]
time values: [0.051107, 0.067859, 0.10999, 0.068076, 0.080347, 0.102238, 0.134826, 0.068076, 0.203353, 0.29869, 0.271889, 0.74062, 1.620886, 3.351269, 6.686959, 14.170136, 22.966939, 40.762055, 62.630188, 94.350882, 140.881171, 221.846619, 313.783617, 533.60272, 773.320634, 1202.683847, 1603.325482, 2126.254855]


In [3]:
import numpy as np

In [5]:
n_and_tau_and_times = np.array([n_values, n_values, time_values])
print(n_and_tau_and_times[0:3:2])
for i in range(len(n_values)):
    print(int(n_and_tau_and_times[0,i]), f"{n_and_tau_and_times[2,i]:.2f} s")
print(" & ".join(n_and_tau_and_times[0,9:].astype(int).astype(str)))
print(" & ".join([f"{time:.2f}" for time in n_and_tau_and_times[2,9:]]))
print(" & ".join([f"{time:.2f}" for time in (n_and_tau_and_times[2,9:] / 60)]))
print("|" + "c|" * len(n_and_tau_and_times[0,9:]))

[[1.00000000e+00 2.00000000e+00 3.00000000e+00 4.00000000e+00
  5.00000000e+00 6.00000000e+00 7.00000000e+00 8.00000000e+00
  9.00000000e+00 1.00000000e+01 1.00000000e+01 1.50000000e+01
  2.00000000e+01 2.50000000e+01 3.00000000e+01 3.50000000e+01
  4.00000000e+01 4.50000000e+01 5.00000000e+01 5.50000000e+01
  6.00000000e+01 6.50000000e+01 7.00000000e+01 7.50000000e+01
  8.00000000e+01 8.50000000e+01 9.00000000e+01 9.50000000e+01]
 [5.11070000e-02 6.78590000e-02 1.09990000e-01 6.80760000e-02
  8.03470000e-02 1.02238000e-01 1.34826000e-01 6.80760000e-02
  2.03353000e-01 2.98690000e-01 2.71889000e-01 7.40620000e-01
  1.62088600e+00 3.35126900e+00 6.68695900e+00 1.41701360e+01
  2.29669390e+01 4.07620550e+01 6.26301880e+01 9.43508820e+01
  1.40881171e+02 2.21846619e+02 3.13783617e+02 5.33602720e+02
  7.73320634e+02 1.20268385e+03 1.60332548e+03 2.12625486e+03]]
1 0.05 s
2 0.07 s
3 0.11 s
4 0.07 s
5 0.08 s
6 0.10 s
7 0.13 s
8 0.07 s
9 0.20 s
10 0.30 s
10 0.27 s
15 0.74 s
20 1.62 s
25 3.35 